# Mode 02 Kaggle: Z24 NPY dataset â€” 1DCNN-LSTM-ResNet

Fast experimental variant: cuDNN-compatible LSTM, mixed precision, batch 32, prefetched `tf.data`, and two-GPU `MirroredStrategy`. Attach the Kaggle Dataset containing `inputs.npy` and `labels.npy`, enable **GPU T4 x2**, then choose **Run All**. Keep the original notebook as the baseline because this variant changes the LSTM recurrent activation from `softmax` to `sigmoid`.

In [ ]:
# ================================================================
# Z24 NPY DATASET -> GROUPED SPLIT -> 1DCNN-LSTM-RESNET -> METRICS
# Self-contained Kaggle notebook. No repository imports are required.
# ================================================================

from pathlib import Path
from datetime import datetime
import gc
import json
import os
import shutil
import zipfile

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from tensorflow.keras.layers import (
    Activation, Add, BatchNormalization, Concatenate, Conv1D, Dense,
    Dropout, GlobalAveragePooling1D, GlobalMaxPooling1D, Input, LSTM,
)
from tensorflow.keras.regularizers import l2


In [ ]:
# --------------------------- Configuration ---------------------------
SEED = 42
EPOCHS = 100
BATCH_SIZE = 32
EARLY_STOPPING_PATIENCE = 15
TRAIN_SETUPS = 6
VALIDATION_SETUPS = 1
EXPECTED_INPUT_SHAPE = (1530, 27, 6000)
EXPECTED_LABEL_SHAPE = (1530,)
NUM_CLASSES = 17
SETUPS_PER_CONDITION = 9
SEGMENTS_PER_RECORDING = 10
KAGGLE_INPUTS_PATH = Path('/kaggle/input/datasets/jenvn37hacker/dataset/inputs.npy')
KAGGLE_LABELS_PATH = Path('/kaggle/input/datasets/jenvn37hacker/dataset/labels.npy')

tf.keras.utils.set_random_seed(SEED)
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
tf.keras.mixed_precision.set_global_policy('mixed_float16' if gpus else 'float32')
strategy = (
    tf.distribute.MirroredStrategy()
    if len(gpus) > 1
    else tf.distribute.get_strategy()
)
print('TensorFlow:', tf.__version__)
print('GPU devices:', gpus if gpus else 'None â€” TensorFlow will use CPU')

print('Mixed precision policy:', tf.keras.mixed_precision.global_policy())
print('Synchronized replicas:', strategy.num_replicas_in_sync)

# Kaggle input is read-only; cache and outputs belong in /kaggle/working.
KAGGLE_WORKING = Path('/kaggle/working')
WORK_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd() / 'kaggle_working'
CACHE_DIR = WORK_DIR / 'z24_dataset_cache'
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
ARTIFACT_DIR = WORK_DIR / f'z24_dcnn_fast_results_{RUN_ID}'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)


In [ ]:
# ---------------------- Locate and prepare data ----------------------
search_roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd(), Path.cwd() / 'raw_data']
search_roots = [path for path in search_roots if path.exists()]
zip_candidates = sorted(
    {candidate.resolve() for root in search_roots for candidate in root.rglob('dataset.zip')},
    key=lambda path: (len(path.parts), str(path)),
)

# The attached Kaggle Dataset shown in the notebook UI uses these exact paths.
if KAGGLE_INPUTS_PATH.is_file() and KAGGLE_LABELS_PATH.is_file():
    inputs_path = KAGGLE_INPUTS_PATH
    labels_path = KAGGLE_LABELS_PATH
    print('Using direct Kaggle NPY files:', inputs_path.parent)
elif zip_candidates:
    archive_path = zip_candidates[0]
    print('Using archive:', archive_path)
    inputs_path = CACHE_DIR / 'inputs.npy'
    labels_path = CACHE_DIR / 'labels.npy'
    with zipfile.ZipFile(archive_path) as archive:
        members = {info.filename: info for info in archive.infolist() if not info.is_dir()}
        for member_name, target in (('inputs.npy', inputs_path), ('labels.npy', labels_path)):
            if member_name not in members:
                raise ValueError(f'{archive_path} is missing {member_name}')
            expected_size = members[member_name].file_size
            if target.exists() and target.stat().st_size != expected_size:
                raise ValueError(f'Cached file has the wrong size: {target}')
            if not target.exists():
                temporary = target.with_suffix(target.suffix + '.partial')
                if temporary.exists():
                    temporary.unlink()
                with archive.open(member_name) as source, temporary.open('wb') as destination:
                    shutil.copyfileobj(source, destination, length=16 * 1024 * 1024)
                temporary.replace(target)
else:
    # Also support a Kaggle Dataset containing the two NPY files directly.
    pair = None
    for root in search_roots:
        for candidate in root.rglob('inputs.npy'):
            sibling = candidate.with_name('labels.npy')
            if sibling.exists() and CACHE_DIR not in candidate.parents:
                pair = (candidate, sibling)
                break
        if pair:
            break
    if pair is None:
        raise FileNotFoundError(
            'dataset.zip was not found. Attach a Kaggle Dataset containing '
            'dataset.zip, or inputs.npy and labels.npy.'
        )
    inputs_path, labels_path = pair
    print('Using direct NPY files:', inputs_path.parent)

inputs = np.load(inputs_path, mmap_mode='r', allow_pickle=False)
labels = np.load(labels_path, allow_pickle=False)
if inputs.shape != EXPECTED_INPUT_SHAPE or inputs.dtype != np.float32:
    raise ValueError(f'Expected float32 inputs {EXPECTED_INPUT_SHAPE}, got {inputs.shape} {inputs.dtype}')
if labels.shape != EXPECTED_LABEL_SHAPE or labels.dtype != np.int64:
    raise ValueError(f'Expected int64 labels {EXPECTED_LABEL_SHAPE}, got {labels.shape} {labels.dtype}')

expected_labels = np.repeat(
    np.arange(NUM_CLASSES, dtype=np.int64),
    SETUPS_PER_CONDITION * SEGMENTS_PER_RECORDING,
)
if not np.array_equal(labels, expected_labels):
    raise ValueError('Labels are not ordered as 17 conditions x 9 setups x 10 segments')
print('Dataset:', inputs.shape, inputs.dtype, '| labels:', labels.shape, labels.dtype)


In [ ]:
# ---------------- Leakage-safe setup-grouped split -----------------
sample_index = np.arange(len(labels), dtype=np.int64)
within_condition = sample_index % (SETUPS_PER_CONDITION * SEGMENTS_PER_RECORDING)
setup_id = within_condition // SEGMENTS_PER_RECORDING
recording_id = labels * SETUPS_PER_CONDITION + setup_id

setup_order = np.arange(SETUPS_PER_CONDITION, dtype=np.int64)
np.random.default_rng(SEED).shuffle(setup_order)
setup_split = {
    'train': setup_order[:TRAIN_SETUPS],
    'validation': setup_order[TRAIN_SETUPS:TRAIN_SETUPS + VALIDATION_SETUPS],
    'test': setup_order[TRAIN_SETUPS + VALIDATION_SETUPS:],
}
split_indexes = {
    name: np.flatnonzero(np.isin(setup_id, selected_setups))
    for name, selected_setups in setup_split.items()
}
recording_sets = {
    name: set(recording_id[indexes].tolist())
    for name, indexes in split_indexes.items()
}
assert recording_sets['train'].isdisjoint(recording_sets['validation'])
assert recording_sets['train'].isdisjoint(recording_sets['test'])
assert recording_sets['validation'].isdisjoint(recording_sets['test'])

for name, indexes in split_indexes.items():
    class_counts = np.bincount(labels[indexes], minlength=NUM_CLASSES)
    assert np.all(class_counts == class_counts[0])
    print(
        f'{name:10s}: {len(indexes):4d} segments, '
        f'{len(recording_sets[name]):3d} recordings, setups={setup_split[name].tolist()}'
    )


In [ ]:
# ------------------- Materialize Keras arrays ----------------------
def materialize_keras(indexes, chunk_size=64):
    result = np.empty((len(indexes), inputs.shape[2], inputs.shape[1]), dtype=np.float32)
    for start in range(0, len(indexes), chunk_size):
        selected = indexes[start:start + chunk_size]
        result[start:start + len(selected)] = np.asarray(
            inputs[selected], dtype=np.float32
        ).transpose(0, 2, 1)
    return result, np.asarray(labels[indexes], dtype=np.int64)

x_train, y_train = materialize_keras(split_indexes['train'])
x_validation, y_validation = materialize_keras(split_indexes['validation'])
x_test, y_test = materialize_keras(split_indexes['test'])

# Per-sensor normalization from train only.
sensor_mean = x_train.mean(axis=(0, 1), keepdims=True, dtype=np.float64).astype(np.float32)
sensor_std = x_train.std(axis=(0, 1), keepdims=True, dtype=np.float64).astype(np.float32)
sensor_std = np.where(sensor_std < 1e-8, 1.0, sensor_std)
for array in (x_train, x_validation, x_test):
    array -= sensor_mean
    array /= sensor_std

for name, X, y in (
    ('train', x_train, y_train),
    ('validation', x_validation, y_validation),
    ('test', x_test, y_test),
):
    assert X.shape[1:] == (6000, 27)
    assert np.isfinite(X).all()
    print(f'{name:10s}: X={X.shape}, y={y.shape}, RAM={X.nbytes / 1024**2:.1f} MiB')

del inputs
gc.collect()


In [ ]:
# -------------------------- Model ----------------------------------
def resnet_block(
    x, filters, kernel_size=3, stride=1, dilation_rate=1,
    use_projection_shortcut=False, use_batch_norm=False,
):
    first_filters, middle_filters, output_filters = filters
    shortcut = x
    x = Conv1D(first_filters, 1, strides=stride, dilation_rate=dilation_rate, padding='valid')(x)
    if use_batch_norm:
        x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv1D(middle_filters, kernel_size, dilation_rate=dilation_rate, padding='same')(x)
    if use_batch_norm:
        x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv1D(output_filters, 1, padding='valid')(x)
    if use_batch_norm:
        x = BatchNormalization()(x)
    if use_projection_shortcut:
        shortcut = Conv1D(output_filters, 1, strides=stride, dilation_rate=dilation_rate, padding='valid')(shortcut)
        if use_batch_norm:
            shortcut = BatchNormalization()(shortcut)
    return Activation('relu')(Add()([x, shortcut]))

def build_model(input_shape=(6000, 27), num_classes=17):
    input_tensor = Input(shape=input_shape)
    x = Conv1D(64, 7, padding='same', strides=2)(input_tensor)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = resnet_block(x, [64, 64, 256], use_projection_shortcut=True)
    x_dilated = resnet_block(
        x, [64, 64, 256], use_projection_shortcut=True, dilation_rate=2
    )
    x = Add()([x, x_dilated])
    lstm = LSTM(128, return_sequences=True, recurrent_activation='sigmoid')(x)
    lstm = Dropout(0.30)(lstm)
    shortcut1 = Conv1D(lstm.shape[-1], 1, padding='same', strides=2)(input_tensor)
    shortcut1 = BatchNormalization()(shortcut1)
    shortcut2 = Conv1D(lstm.shape[-1], 1, padding='same')(x)
    shortcut2 = BatchNormalization()(shortcut2)
    x = Concatenate(axis=-1)([lstm, shortcut1, shortcut2])
    x = Activation('relu')(x)
    x = Concatenate(axis=-1)([GlobalAveragePooling1D()(x), GlobalMaxPooling1D()(x)])
    x = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.50)(x)
    output_tensor = Dense(num_classes, activation='softmax', dtype='float32')(x)
    model = tf.keras.Model(input_tensor, output_tensor)
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

tf.keras.backend.clear_session()
with strategy.scope():
    model = build_model()
model.summary()


In [ ]:
# -------------------------- Training -------------------------------
AUTOTUNE = tf.data.AUTOTUNE
data_options = tf.data.Options()
data_options.experimental_deterministic = False
train_dataset = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(len(x_train), seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
    .with_options(data_options)
)
validation_dataset = (
    tf.data.Dataset.from_tensor_slices((x_validation, y_validation))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
print('Training batches per epoch:', int(np.ceil(len(x_train) / BATCH_SIZE)))
print('Validation batches:', int(np.ceil(len(x_validation) / BATCH_SIZE)))

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        ARTIFACT_DIR / 'best_model.keras', monitor='val_loss', mode='min', save_best_only=True
    ),
]

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
# ---------------- Train / validation / test metrics ----------------
def evaluate_split(split_name, X, y):
    probabilities = model.predict(X, batch_size=BATCH_SIZE, verbose=0)
    predictions = probabilities.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y, predictions, average='macro', zero_division=0
    )
    return {
        'split': split_name,
        'accuracy': accuracy_score(y, predictions),
        'precision_macro': precision,
        'recall_macro': recall,
        'f1_macro': f1,
    }, predictions

metric_rows = []
predictions_by_split = {}
for split_name, X, y in (
    ('train', x_train, y_train),
    ('validation', x_validation, y_validation),
    ('test', x_test, y_test),
):
    row, split_predictions = evaluate_split(split_name, X, y)
    metric_rows.append(row)
    predictions_by_split[split_name] = split_predictions

split_metrics = pd.DataFrame(metric_rows).set_index('split')
print()
print('Final metrics - precision/recall/F1 are macro averages:')
display(split_metrics.style.format('{:.2%}'))

test_predictions = predictions_by_split['test']
test_report = classification_report(
    y_test, test_predictions, labels=np.arange(NUM_CLASSES),
    output_dict=True, zero_division=0,
)
test_matrix = confusion_matrix(y_test, test_predictions, labels=np.arange(NUM_CLASSES))


In [ ]:
# ----------------------- Save all outputs --------------------------
model.save(ARTIFACT_DIR / 'final_model.keras')
pd.DataFrame(history.history).to_csv(ARTIFACT_DIR / 'history.csv', index=False)
split_metrics.to_csv(ARTIFACT_DIR / 'split_metrics.csv')
pd.DataFrame(test_report).T.to_csv(ARTIFACT_DIR / 'test_classification_report.csv')
np.savetxt(ARTIFACT_DIR / 'test_confusion_matrix.csv', test_matrix, fmt='%d', delimiter=',')
np.savez(
    ARTIFACT_DIR / 'preprocessing_and_splits.npz',
    sensor_mean=sensor_mean.reshape(-1),
    sensor_std=sensor_std.reshape(-1),
    train_indexes=split_indexes['train'],
    validation_indexes=split_indexes['validation'],
    test_indexes=split_indexes['test'],
)

experiment = {
    'seed': SEED,
    'epochs_requested': EPOCHS,
    'epochs_completed': len(history.history['loss']),
    'batch_size': BATCH_SIZE,
    'mixed_precision_policy': tf.keras.mixed_precision.global_policy().name,
    'replicas': strategy.num_replicas_in_sync,
    'lstm_recurrent_activation': 'sigmoid',
    'stored_shape': list(EXPECTED_INPUT_SHAPE),
    'model_input_shape': [6000, 27],
    'num_classes': NUM_CLASSES,
    'setup_split': {name: values.tolist() for name, values in setup_split.items()},
    'gpu_devices': [device.name for device in gpus],
    'tensorflow_version': tf.__version__,
    'metrics': split_metrics.to_dict(orient='index'),
}
(ARTIFACT_DIR / 'experiment.json').write_text(
    json.dumps(experiment, indent=2), encoding='utf-8'
)

history_frame = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
history_frame[['loss', 'val_loss']].plot(ax=axes[0], title='Loss')
history_frame[['accuracy', 'val_accuracy']].plot(ax=axes[1], title='Accuracy')
for axis in axes:
    axis.set_xlabel('Epoch')
    axis.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'training_curves.png', dpi=160, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay(test_matrix, display_labels=np.arange(NUM_CLASSES)).plot(
    ax=ax, cmap='Blues', colorbar=False
)
ax.set_title('Mode 02 FAST test confusion matrix')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'test_confusion_matrix.png', dpi=160, bbox_inches='tight')
plt.show()

archive_output = shutil.make_archive(str(ARTIFACT_DIR), 'zip', root_dir=ARTIFACT_DIR)
print()
print('Completed successfully.')
print('Artifacts:', ARTIFACT_DIR)
print('Download ZIP:', archive_output)